# UniProt API Exploration: Cancer-Related Proteins

## API Questions

- Organism filter: organism_id:9606
- Search term: cancer
- Result size for first test: 10

Query Formulierung: organism_id:9606 AND cancer AND reviewed:true

Output = 'https://rest.uniprot.org/uniprotkb/search?query=organism%3A9606%20AND%20cancer%20AND%20reviewed%3Atrue&size=10' 

## Notes

- I restrict the first API test to reviewed human protein entries because reviewed UniProt records are manually curated and easier to interpret for an initial ETL prototype.

- The API request returned status code 200, which means that the request was successful and the UniProt server returned a valid response (400 = Bad Request. 401 = Unauthorized. 404 = Not Found. 500 = Server Error)

Keys of interest:
1. accession
2. protein name
3. gene name
4. organism scientific name
5. sequence length

## Modules


In [2]:
import requests
import json
import sys
import pprint

## Get Request

In [3]:
base_url = "https://rest.uniprot.org/uniprotkb/search"

#Parameters and Query
organism = "organism_id:9606" #=human
search_term = "cancer"
size = 10

query = f'{organism} AND {search_term} AND reviewed%3Atrue&size={size}'

#URL
get_url = f'{base_url}?query={query}'
response = requests.get(get_url)

if response.status_code == 200:
    dataset = response.json()
else:
    print(f'Error at request: {response}')

#API Test
print(f'''
response.status_code = {response.status_code}
dataset.keys() = {dataset.keys()}
len(dataset["results"]) = {len(dataset["results"])}
dataset["results"][0].keys() = {dataset["results"][0].keys()}
''')


response.status_code = 200
dataset.keys() = dict_keys(['results'])
len(dataset["results"]) = 10
dataset["results"][0].keys() = dict_keys(['entryType', 'primaryAccession', 'secondaryAccessions', 'uniProtkbId', 'entryAudit', 'annotationScore', 'organism', 'proteinExistence', 'proteinDescription', 'genes', 'comments', 'features', 'keywords', 'references', 'uniProtKBCrossReferences', 'sequence', 'extraAttributes'])



In [36]:
record = dataset["results"]

#Keys of interest
print(f'''
Primary Accession:          {record[0]['primaryAccession']}
Protein Name:               {record[0]['proteinDescription']['alternativeNames'][0]['shortNames'][0]['value']}
Gene Name:                  {record[0]['genes'][0]['geneName']['value']}
Organism Scientific Name:   {record[0]['organism']['scientificName']}
Sequence Length:            {record[0]['sequence']['length']}
''')



Primary Accession:          Q9Y238
Protein Name:               DLC-1
Gene Name:                  DLEC1
Organism Scientific Name:   Homo sapiens
Sequence Length:            1755



In [37]:
flat_record = {
    "accession":record[0]['primaryAccession'],
    "protein_name":record[0]['proteinDescription']['alternativeNames'][0]['shortNames'][0]['value'],
    "gene_name" : record[0]['genes'][0]['geneName']['value'],
    "organism_scientific_name":record[0]['organism']['scientificName'],
    "sequence_length": record[0]['sequence']['length']
}

flat_record

{'accession': 'Q9Y238',
 'protein_name': 'DLC-1',
 'gene_name': 'DLEC1',
 'organism_scientific_name': 'Homo sapiens',
 'sequence_length': 1755}

In [43]:
flat_records = []

for i in dataset["results"]:

    protein_name = None

    try:
        protein_name = (i["proteinDescription"]["alternativeNames"][0]["shortNames"][0]["value"])
    except (KeyError, IndexError, TypeError):
        protein_name = None

    gene_name = None

    try:
        gene_name = i["genes"][0]["geneName"]["value"]
    except (KeyError, IndexError, TypeError):
        gene_name = None

    flat_record = {
        "Accession": i.get("primaryAccession"),
        "Protein Name": protein_name,
        "Gene Name": gene_name,
        "Organism Scientific Name": i.get("organism", {}).get("scientificName"),
        "Sequence Length": i.get("sequence", {}).get("length"),
    }

    flat_records.append(flat_record)

flat_records

    


[{'Accession': 'Q9Y238',
  'Protein Name': 'DLC-1',
  'Gene Name': 'DLEC1',
  'Organism Scientific Name': 'Homo sapiens',
  'Sequence Length': 1755},
 {'Accession': 'P51587',
  'Protein Name': None,
  'Gene Name': 'BRCA2',
  'Organism Scientific Name': 'Homo sapiens',
  'Sequence Length': 3418},
 {'Accession': 'Q5HYN5',
  'Protein Name': None,
  'Gene Name': 'CT45A1',
  'Organism Scientific Name': 'Homo sapiens',
  'Sequence Length': 189},
 {'Accession': 'O00559',
  'Protein Name': None,
  'Gene Name': 'EBAG9',
  'Organism Scientific Name': 'Homo sapiens',
  'Sequence Length': 213},
 {'Accession': 'P38398',
  'Protein Name': None,
  'Gene Name': 'BRCA1',
  'Organism Scientific Name': 'Homo sapiens',
  'Sequence Length': 1863},
 {'Accession': 'Q8TC20',
  'Protein Name': 'CT3',
  'Gene Name': 'CAGE1',
  'Organism Scientific Name': 'Homo sapiens',
  'Sequence Length': 777},
 {'Accession': 'Q9HCU9',
  'Protein Name': None,
  'Gene Name': 'BRMS1',
  'Organism Scientific Name': 'Homo sapiens